# Step 3: Feature Extraction

In [1]:
# Import libraries and load the cleaned URL dataset

from pathlib import Path
import pandas as pd
import re
import math
import ipaddress
from urllib.parse import urlsplit, parse_qs

current_folder = Path.cwd().resolve()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

cleaned_dataset_path = project_root / "data" / "cleaned" / "cleaned_urls.csv"

df_clean = pd.read_csv(cleaned_dataset_path)

print("Cleaned dataset loaded successfully.")
print(f"Dataset shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

display(df_clean.head())

Cleaned dataset loaded successfully.
Dataset shape: 253,098 rows × 2 columns


,url,label
0,https://www.visitcanada.com,legitimate
1,http://218.228.19.9/~yossi/9ssfpkz,phishing
2,https://www.msupress.msu.edu/series.php?series...,legitimate
3,https://docs.google.com/presentation/d/e/2PACX...,phishing
4,https://www.c250.columbia.edu/c250_celebrates/...,legitimate


In [2]:
# Define fixed lists and thresholds for planned URL features

SUSPICIOUS_TLDS = {
    "zip", "mov", "top", "xyz", "tk", "gq", "ml", "cf",
    "work", "click", "link", "country", "stream", "download",
    "science", "racing", "review", "party", "monster", "cam"
}

URL_SHORTENING_DOMAINS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly",
    "is.gd", "buff.ly", "cutt.ly", "rb.gy", "shorturl.at",
    "tiny.cc", "rebrand.ly", "shorte.st", "adf.ly"
}

SUSPICIOUS_KEYWORDS = {
    "login", "signin", "verify", "verification", "account",
    "secure", "security", "update", "confirm", "password",
    "banking", "wallet", "payment", "invoice", "recover",
    "suspend", "unlock", "validate", "credential"
}

BRAND_NAMES = {
    "google", "paypal", "microsoft", "apple", "amazon",
    "facebook", "instagram", "netflix", "whatsapp", "yahoo",
    "linkedin", "dropbox", "bankofamerica", "chase"
}

EXCESSIVE_HYPHEN_THRESHOLD = 3

print("Feature rules defined successfully.")
print(f"Suspicious TLDs: {len(SUSPICIOUS_TLDS)}")
print(f"URL-shortening domains: {len(URL_SHORTENING_DOMAINS)}")
print(f"Suspicious keywords: {len(SUSPICIOUS_KEYWORDS)}")
print(f"Brand names: {len(BRAND_NAMES)}")
print(f"Excessive-hyphen threshold: more than {EXCESSIVE_HYPHEN_THRESHOLD} hyphens")

Feature rules defined successfully.
Suspicious TLDs: 20
URL-shortening domains: 14
Suspicious keywords: 19
Brand names: 14
Excessive-hyphen threshold: more than 3 hyphens


In [3]:
# Create helper functions for URL parsing and entropy calculation

from collections import Counter
from urllib.parse import unquote

def calculate_entropy(value):
    if not value:
        return 0.0

    counts = Counter(value)
    total = len(value)

    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values()
    )


def get_url_parts(url):
    parsed = urlsplit(str(url))

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""
    scheme = parsed.scheme.lower()

    return parsed, hostname.lower(), path, query, scheme


def is_ip_address(hostname):
    try:
        ipaddress.ip_address(hostname)
        return 1
    except ValueError:
        return 0


def count_letters(value):
    return sum(character.isalpha() for character in value)


def count_digits(value):
    return sum(character.isdigit() for character in value)


def count_query_parameters(query):
    if not query:
        return 0

    return len(parse_qs(query, keep_blank_values=True))


print("Helper functions created successfully.")

Helper functions created successfully.


In [4]:
# Create the function that extracts the fixed 28 planned features

def extract_features(url):
    url = str(url)
    parsed, hostname, path, query, scheme = get_url_parts(url)

    url_lower = url.lower()
    hostname_lower = hostname.lower()

    url_length = len(url)
    num_dots = url.count(".")
    num_slashes = url.count("/")
    num_hyphens = url.count("-")
    num_underscores = url.count("_")
    num_special_characters = sum(
        not character.isalnum()
        for character in url
    )
    num_digits = count_digits(url)
    num_letters = count_letters(url)

    digit_to_letter_ratio = (
        num_digits / num_letters
        if num_letters > 0
        else 0.0
    )

    path_length = len(path)
    query_string_length = len(query)
    num_query_parameters = count_query_parameters(query)
    num_at_symbols = url.count("@")
    num_double_slash_occurrences = url.count("//")
    url_entropy = calculate_entropy(url)

    domain_length = len(hostname)

    domain_parts = [
        part for part in hostname.split(".")
        if part
    ]

    num_subdomains = max(len(domain_parts) - 2, 0)

    tld_type = (
        domain_parts[-1]
        if domain_parts and not is_ip_address(hostname)
        else "ip"
    )

    suspicious_tld = int(tld_type in SUSPICIOUS_TLDS)
    punycode_idn_usage = int("xn--" in hostname_lower)
    ip_address_instead_of_domain = is_ip_address(hostname)
    https_usage = int(scheme == "https")

    url_shortening_service = int(
        hostname_lower in URL_SHORTENING_DOMAINS
        or any(
            hostname_lower.endswith("." + domain)
            for domain in URL_SHORTENING_DOMAINS
        )
    )

    encoded_characters = url.count("%")

    hexadecimal_characters = len(
        re.findall(r"(?i)(?:0x[0-9a-f]+|[0-9a-f]{2})", url)
    )

    suspicious_keywords = int(
        any(keyword in url_lower for keyword in SUSPICIOUS_KEYWORDS)
    )

    brand_name_presence = int(
        any(brand in url_lower for brand in BRAND_NAMES)
    )

    excessive_hyphens = int(
        num_hyphens > EXCESSIVE_HYPHEN_THRESHOLD
    )

    hostname_without_common_prefix = re.sub(
        r"^(www\d?\.)",
        "",
        hostname_lower
    )

    hostname_alphanumeric = re.sub(
        r"[^a-z0-9]",
        "",
        hostname_without_common_prefix
    )

    random_looking_domain = int(
        len(hostname_alphanumeric) >= 12
        and count_digits(hostname_without_common_prefix) >= 3
        and calculate_entropy(hostname_alphanumeric) >= 3.5
    )

    return {
        "url_length": url_length,
        "num_dots": num_dots,
        "num_slashes": num_slashes,
        "num_hyphens": num_hyphens,
        "num_underscores": num_underscores,
        "num_special_characters": num_special_characters,
        "num_digits": num_digits,
        "digit_to_letter_ratio": digit_to_letter_ratio,
        "path_length": path_length,
        "query_string_length": query_string_length,
        "num_query_parameters": num_query_parameters,
        "num_at_symbols": num_at_symbols,
        "num_double_slash_occurrences": num_double_slash_occurrences,
        "url_entropy": url_entropy,
        "domain_length": domain_length,
        "num_subdomains": num_subdomains,
        "tld_type": tld_type,
        "suspicious_tld": suspicious_tld,
        "punycode_idn_usage": punycode_idn_usage,
        "ip_address_instead_of_domain": ip_address_instead_of_domain,
        "https_usage": https_usage,
        "url_shortening_service": url_shortening_service,
        "encoded_characters": encoded_characters,
        "hexadecimal_characters": hexadecimal_characters,
        "suspicious_keywords": suspicious_keywords,
        "brand_name_presence": brand_name_presence,
        "excessive_hyphens": excessive_hyphens,
        "random_looking_domain": random_looking_domain
    }


print("Feature extraction function created successfully.")

Feature extraction function created successfully.


In [5]:
# Test feature extraction on one URL before processing the full dataset

test_url = df_clean.loc[0, "url"]
test_features = extract_features(test_url)

print("Test URL:")
print(test_url)

print("\nExtracted feature count:")
print(len(test_features))

print("\nExtracted features:")
display(pd.DataFrame([test_features]))

Test URL:
https://www.visitcanada.com

Extracted feature count:
28

Extracted features:


,url_length,num_dots,num_slashes,num_hyphens,num_underscores,num_special_characters,num_digits,digit_to_letter_ratio,path_length,query_string_length,...,punycode_idn_usage,ip_address_instead_of_domain,https_usage,url_shortening_service,encoded_characters,hexadecimal_characters,suspicious_keywords,brand_name_presence,excessive_hyphens,random_looking_domain
0,27,2,2,0,0,5,0,0.0,0,0,...,0,0,1,0,0,2,0,0,0,0


In [6]:
# Verify that the extractor returns exactly the 28 planned features

expected_feature_names = [
    "url_length",
    "num_dots",
    "num_slashes",
    "num_hyphens",
    "num_underscores",
    "num_special_characters",
    "num_digits",
    "digit_to_letter_ratio",
    "path_length",
    "query_string_length",
    "num_query_parameters",
    "num_at_symbols",
    "num_double_slash_occurrences",
    "url_entropy",
    "domain_length",
    "num_subdomains",
    "tld_type",
    "suspicious_tld",
    "punycode_idn_usage",
    "ip_address_instead_of_domain",
    "https_usage",
    "url_shortening_service",
    "encoded_characters",
    "hexadecimal_characters",
    "suspicious_keywords",
    "brand_name_presence",
    "excessive_hyphens",
    "random_looking_domain"
]

actual_feature_names = list(test_features.keys())

print("Expected feature count:", len(expected_feature_names))
print("Actual feature count:", len(actual_feature_names))
print("Feature names match:", expected_feature_names == actual_feature_names)

missing_features = set(expected_feature_names) - set(actual_feature_names)
extra_features = set(actual_feature_names) - set(expected_feature_names)

print("Missing features:", missing_features)
print("Extra features:", extra_features)

Expected feature count: 28
Actual feature count: 28
Feature names match: True
Missing features: set()
Extra features: set()


In [7]:
# Extract all 28 features from every URL

feature_records = [
    extract_features(url)
    for url in df_clean["url"]
]

df_features = pd.DataFrame(feature_records)

df_features.insert(0, "url", df_clean["url"].values)
df_features.insert(1, "label", df_clean["label"].values)

print("Feature extraction completed.")
print(f"Feature dataset shape: {df_features.shape[0]:,} rows × {df_features.shape[1]:,} columns")
display(df_features.head())

Feature extraction completed.
Feature dataset shape: 253,098 rows × 30 columns


,url,label,url_length,num_dots,num_slashes,num_hyphens,num_underscores,num_special_characters,num_digits,digit_to_letter_ratio,...,punycode_idn_usage,ip_address_instead_of_domain,https_usage,url_shortening_service,encoded_characters,hexadecimal_characters,suspicious_keywords,brand_name_presence,excessive_hyphens,random_looking_domain
0,https://www.visitcanada.com,legitimate,27,2,2,0,0,5,0,0.000000,...,0,0,1,0,0,2,0,0,0,0
1,http://218.228.19.9/~yossi/9ssfpkz,phishing,34,3,4,0,0,9,10,0.666667,...,0,1,0,0,0,3,0,0,0,0
2,https://www.msupress.msu.edu/series.php?series...,legitimate,51,4,3,0,0,10,2,0.051282,...,0,0,1,0,0,2,0,0,0,0
3,https://docs.google.com/presentation/d/e/2PACX...,phishing,175,2,7,2,1,21,19,0.140741,...,0,0,1,0,0,12,0,1,0,0
4,https://www.c250.columbia.edu/c250_celebrates/...,legitimate,79,4,5,0,2,12,6,0.098361,...,0,0,1,0,0,7,0,0,0,1


In [10]:
# Check feature data types, missing values, and infinite values

import numpy as np

print("Missing values in extracted features:")
display(df_features[feature_columns].isna().sum().to_frame(name="missing_count"))

print("\nInfinite values in numeric features:")
numeric_feature_columns = [
    column for column in feature_columns
    if column != "tld_type"
]

infinite_counts = pd.Series({
    column: np.isinf(
        pd.to_numeric(df_features[column], errors="coerce")
    ).sum()
    for column in numeric_feature_columns
})

display(infinite_counts.to_frame(name="infinite_count"))

print("\nFeature data types:")
display(df_features[feature_columns].dtypes.to_frame(name="data_type"))

Missing values in extracted features:


,missing_count
url_length,0
num_dots,0
num_slashes,0
num_hyphens,0
num_underscores,0
num_special_characters,0
num_digits,0
digit_to_letter_ratio,0
path_length,0
query_string_length,0



Infinite values in numeric features:


,infinite_count
url_length,0
num_dots,0
num_slashes,0
num_hyphens,0
num_underscores,0
num_special_characters,0
num_digits,0
digit_to_letter_ratio,0
path_length,0
query_string_length,0



Feature data types:


,data_type
url_length,int64
num_dots,int64
num_slashes,int64
num_hyphens,int64
num_underscores,int64
num_special_characters,int64
num_digits,int64
digit_to_letter_ratio,float64
path_length,int64
query_string_length,int64


In [11]:
# Save the extracted feature dataset and feature definitions

processed_data_folder = project_root / "data" / "processed"
results_logs_folder = project_root / "results" / "logs"

processed_data_folder.mkdir(parents=True, exist_ok=True)
results_logs_folder.mkdir(parents=True, exist_ok=True)

processed_features_path = processed_data_folder / "url_features.csv"
feature_definitions_path = results_logs_folder / "feature_definitions.csv"

feature_definitions = pd.DataFrame({
    "feature_number": list(range(1, 29)),
    "feature_name": expected_feature_names,
    "description": [
        "Total URL character count",
        "Count of period characters",
        "Count of slash characters",
        "Count of hyphen characters",
        "Count of underscore characters",
        "Count of non-alphanumeric characters",
        "Count of numeric characters",
        "Digits divided by alphabetic characters",
        "URL path character count",
        "Query-string character count",
        "Number of parsed query parameters",
        "Count of at symbols",
        "Count of double-slash occurrences",
        "Shannon entropy of the URL",
        "Hostname character count",
        "Number of hostname subdomains",
        "Final hostname label",
        "Whether the TLD is in the suspicious-TLD list",
        "Whether the hostname contains xn--",
        "Whether the hostname is an IP address",
        "Whether the scheme is HTTPS",
        "Whether the hostname matches a shortening-service domain",
        "Count of percent characters",
        "Count of hexadecimal-looking sequences",
        "Whether a suspicious keyword occurs",
        "Whether a listed brand name occurs",
        "Whether hyphen count exceeds the threshold",
        "Whether the hostname meets the random-looking rule"
    ]
})

df_features.to_csv(processed_features_path, index=False)
feature_definitions.to_csv(feature_definitions_path, index=False)

print("Extracted feature dataset saved to:")
print(processed_features_path)

print("\nFeature definitions saved to:")
print(feature_definitions_path)

Extracted feature dataset saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\data\processed\url_features.csv

Feature definitions saved to:
C:\Users\Jiya\Documents\Malicious URL Detection\results\logs\feature_definitions.csv


In [12]:
# Import the reusable feature-extraction function

import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
sys.path.append(str(project_root))

from src.features.feature_extractor import extract_features

In [13]:
# Extract features from one URL

features = extract_features("https://example.com/login")
features

{'url_length': 25,
 'num_dots': 1,
 'num_slashes': 3,
 'num_hyphens': 0,
 'num_underscores': 0,
 'num_special_characters': 5,
 'num_digits': 0,
 'digit_to_letter_ratio': 0.0,
 'path_length': 6,
 'query_string_length': 0,
 'num_query_parameters': 0,
 'num_at_symbols': 0,
 'num_double_slash_occurrences': 1,
 'url_entropy': 3.973660689688186,
 'domain_length': 11,
 'num_subdomains': 0,
 'tld_type': 'com',
 'suspicious_tld': 0,
 'punycode_idn_usage': 0,
 'ip_address_instead_of_domain': 0,
 'https_usage': 1,
 'url_shortening_service': 0,
 'encoded_characters': 0,
 'hexadecimal_characters': 0,
 'suspicious_keywords': 1,
 'brand_name_presence': 0,
 'excessive_hyphens': 0,
 'random_looking_domain': 0}